# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Aishwarya00608/FlyRank_Assignment1/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

# Week 06: Validation Audit & Methodological Integrity

## 1. Two Paper Findings & Methodological Questions

### Finding 1: Refresh Impact on Organic Clicks
* **Paper Claim:** Content refreshed based on algorithmic freshness signals showed a measured increase in organic traffic compared to unrefreshed baseline pages.
* **Methodology Question:** *How was the counterfactual label established without survivorship bias?*
  * Specifically, were refreshed articles systematically different prior to intervention (e.g., higher historic authority or active editorial sponsorship)? If pages chosen for refreshes already possessed higher historical impression baselines, the observed traffic lift could partially reflect selection bias rather than the causal impact of the refresh signal alone.

### Finding 2: Cross-Domain Model Generalizability
* **Paper Claim:** The predictive ranking model maintained consistent accuracy across diverse publishing verticals and client domains.
* **Methodology Question:** *Does the validation split isolate domain clusters, or did client tokens bleed across folds?*
  * If the validation set shared `client_hash_id` entities with the training set, the model may have learned client-specific topical authority baselines (memorizing which domains consistently rank on Page 1) rather than generalizable content-quality features. A grouped or leave-one-client-out split is necessary to confirm true cross-domain portability.

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

## 2. My Model Under an Honest Split (Before vs. After)

We compare two validation designs on the identical model (`RandomForestClassifier`, `max_depth=6`):
1. **Before (Naive Random Split):** Rows randomly assigned to train/test. Allows identical clients and similar content clusters to populate both sets simultaneously, artificially boosting metrics.
2. **After (Honest Grouped Split):** Entire client organizations (`client_hash_id`) are held out completely. Tests whether the model generalizes to a new domain without historical client data.

In [5]:
# Build reproducible sample frame with honest, non-leaking features
df_sample = con.execute(f"""
    SELECT
        content_hash_id,
        client_hash_id,
        -- Target
        CASE WHEN {pos_col} <= 10.0 THEN 1 ELSE 0 END AS target_page_one,

        -- Honest Feature 1: Log impressions
        LN(COALESCE({imp_col}, 0) + 1.0) AS log_impressions,

        -- Honest Feature 2: Historical CTR
        (COALESCE({click_col}, 0) * 1.0 / (COALESCE({imp_col}, 0) + 1.0)) AS historical_ctr,

        -- Honest Feature 3: Social traffic share
        (COALESCE({sess_soc}, 0) * 1.0 / (COALESCE({sess_tot}, 0) + 1.0)) AS social_share_ratio,

        -- Honest Feature 4: Scroll engagement
        (COALESCE({scrolls}, 0) * 1.0 / (COALESCE({sess_tot}, 0) + 1.0)) AS scrolls_per_session,

        -- Honest Feature 5: Content ID complexity / URL Depth proxy
        LENGTH(content_hash_id) - LENGTH(REPLACE(content_hash_id, '_', '')) AS structure_depth
    FROM df_raw
    WHERE gsc_data_available IS TRUE AND {imp_col} >= 10
    USING SAMPLE 50000
""").df()

feature_cols = [
    "log_impressions",
    "historical_ctr",
    "social_share_ratio",
    "scrolls_per_session",
    "structure_depth"
]
X = df_sample[feature_cols]
y = df_sample["target_page_one"]
groups = df_sample["client_hash_id"]

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

## 3. Leakage Audit & Concrete Error Examples

### Automated Leakage Check
We test every feature against the label for temporal contamination or mathematical dependency:
* Check for mutual information or linear correlation thresholds exceeding $|r| > 0.85$.
* Ensure all aggregate metrics only pool data strictly preceding or within the March observation boundary.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

def evaluate_split(X_tr, y_tr, X_te, y_te):
    clf = RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42, n_jobs=-1)
    clf.fit(X_tr, y_tr)
    probs = clf.predict_proba(X_te)[:, 1]
    cutoff = np.percentile(probs, 80)
    preds_binary = (probs >= cutoff).astype(int)
    return {
        "ROC-AUC": round(float(roc_auc_score(y_te, probs)), 4),
        "Precision@Top20%": round(float(precision_score(y_te, preds_binary, zero_division=0)), 4),
        "Recall@Top20%": round(float(recall_score(y_te, preds_binary, zero_division=0)), 4),
        "Brier Score": round(float(brier_score_loss(y_te, probs)), 4)
    }, clf, probs

# 1. Naive Random Split (Before)
X_train_rand, X_test_rand, y_train_rand, y_test_rand = train_test_split(
    X, y, test_size=0.20, random_state=42
)
metrics_naive, clf_naive, _ = evaluate_split(X_train_rand, y_train_rand, X_test_rand, y_test_rand)

# 2. Honest Grouped Split by client_hash_id (After)
gss = GroupShuffleSplit(n_splits=1, train_size=0.80, random_state=42)
tr_idx, te_idx = next(gss.split(X, y, groups=groups))

X_train_grp, X_test_grp = X.iloc[tr_idx], X.iloc[te_idx]
y_train_grp, y_test_grp = y.iloc[tr_idx], y.iloc[te_idx]
metrics_grouped, clf_honest, honest_probs = evaluate_split(X_train_grp, y_train_grp, X_test_grp, y_test_grp)

# Comparison Table
split_comparison = pd.DataFrame({
    "Naive Random Split (Leaked Domain Context)": metrics_naive,
    "Honest Grouped Split (Client Held-Out)": metrics_grouped
}).T

print("=== Split Design Audit: Before vs. After ===")
print(split_comparison.to_string())

# Save audit receipt
os.makedirs("../outputs", exist_ok=True)
with open("../outputs/w06_split_audit.json", "w") as f:
    json.dump({
        "naive_split": metrics_naive,
        "grouped_split": metrics_grouped
    }, f, indent=2)

=== Split Design Audit: Before vs. After ===
                                            ROC-AUC  Precision@Top20%  Recall@Top20%  Brier Score
Naive Random Split (Leaked Domain Context)   0.6201            0.7661         0.2412       0.2218
Honest Grouped Split (Client Held-Out)       0.6314            0.8333         0.2467       0.2077


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

## 4. Claim Rewrite: Moving to Safe, Decision-Support Language

### 1. Concrete Error Findings
* **False Positives (e.g., `content_4fdb9cd60244859a`):** Predicted with 90.1% probability of Page-1 placement due to high impression volume (`log_impressions` = 6.79), but failed to rank in the top 10. The model over-indexed on raw demand signals while lacking external competitor domain authority data.
* **False Negatives (e.g., `content_918eb0537ea3a8b8`):** Scored with only 12.5% probability due to lower historical volume (`log_impressions` = 4.98), yet successfully ranked in the top 10. This indicates niche or long-tail topics can rank effectively without requiring high historical engagement baselines.

### 2. Rewritten Claims Using Safe Language
* ❌ **Original Overclaim:** *"The model accurately identifies all high-opportunity content with 90% confidence."*
  *  **Safe Decision-Support Claim:** *"In the March 2026 panel under an honest held-out client split, the model demonstrated directional utility for prioritizing review queues, but exhibited false-positive tendencies on high-impression items that encounter unmodeled competitive resistance."*
* ❌ **Original Overclaim:** *"Impression volume is the primary driver of page-one rankings."*
  *  **Safe Decision-Support Claim:** *"A moderate positive association was observed between log impressions and ranking outcomes, but volume alone is not a sufficient causal predictor of page-one presence."*

In [7]:
# 1. Feature Correlation / Leakage Check
correlations = X_train_grp.apply(lambda col: col.corr(y_train_grp)).round(4)
print("=== Feature-Target Correlation Audit ===")
print(correlations.to_string())

# Flag columns with zero-variance (NaN) or degenerate correlation (|r| >= 0.85)
for feat, corr_val in correlations.items():
    if pd.isna(corr_val):
        print(f"⚠️ Note: {feat} has constant/zero variance in this split (corr=NaN).")
    else:
        assert abs(corr_val) < 0.85, f"🚨 Leakage detected in {feat}: corr={corr_val}"

print("\n✅ Leakage Audit Passed: No feature exhibits degenerate target correlation (|r| >= 0.85).")

# 2. Extract Concrete Error Examples from the Honest Split
test_audit = df_sample.iloc[te_idx].copy()
test_audit["predicted_prob"] = honest_probs
test_audit["predicted_class"] = (honest_probs >= np.percentile(honest_probs, 80)).astype(int)

# False Positives: Confidently predicted Page 1, but actually deep
fp_examples = test_audit[
    (test_audit["target_page_one"] == 0) & (test_audit["predicted_class"] == 1)
].sort_values(by="predicted_prob", ascending=False).head(3)

# False Negatives: Predicted deep, but actually reached Page 1
fn_examples = test_audit[
    (test_audit["target_page_one"] == 1) & (test_audit["predicted_class"] == 0)
].sort_values(by="predicted_prob", ascending=True).head(3)

cols_to_view = ["content_hash_id", "client_hash_id", "log_impressions", "predicted_prob", "target_page_one"]

print("\n=== Audit: Concrete False Positive Failures ===")
print(fp_examples[cols_to_view].to_string(index=False))

print("\n=== Audit: Concrete False Negative Failures ===")
print(fn_examples[cols_to_view].to_string(index=False))

=== Feature-Target Correlation Audit ===
log_impressions        0.0638
historical_ctr         0.0733
social_share_ratio    -0.0145
scrolls_per_session   -0.0599
structure_depth           NaN
⚠️ Note: structure_depth has constant/zero variance in this split (corr=NaN).

✅ Leakage Audit Passed: No feature exhibits degenerate target correlation (|r| >= 0.85).

=== Audit: Concrete False Positive Failures ===
         content_hash_id          client_hash_id  log_impressions  predicted_prob  target_page_one
content_4fdb9cd60244859a client_fef1a8f436438636         6.786717        0.901389                0
content_181bb151a544dec4 client_fef1a8f436438636         6.647688        0.836061                0
content_6b5c4c849789343f client_fef1a8f436438636         6.740519        0.814206                0

=== Audit: Concrete False Negative Failures ===
         content_hash_id          client_hash_id  log_impressions  predicted_prob  target_page_one
content_f10a9428d15aff86 client_fef1a8f436438636

/usr/local/lib/python3.13/dist-packages/numpy/lib/_function_base_impl.py:2999: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/usr/local/lib/python3.13/dist-packages/numpy/lib/_function_base_impl.py:3000: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]


## Self-check

Before you submit, confirm each line honestly:

- [✅] Every section above is filled — markdown thinking AND the code that backs it
- [✅] The notebook runs top to bottom with no errors (Runtime → Run all)
- [✅] No client names, URLs, or private queries anywhere
- [✅] My claims use careful words: observed, measured, directional, decision-support
- [✅] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.